# Demand Prediction Agent
This notebook models future charging demand and station utilization using XGBoost.
We chose XGBoost because it excels at tabular data with non-linear relationships 
and can handle the engineered temporal features robustly.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os
import joblib

# Paths
base_dir = r"c:\Users\SIDDHARTHA\Downloads\Socbiz"
clean_dir = os.path.join(base_dir, "Cleaned_Data")
out_dir = os.path.join(base_dir, "Outputs")
model_dir = os.path.join(base_dir, "Models")
os.makedirs(out_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)

print("Loading cleaned dataset for modeling (using UrbanEV for continuous time-series modeling)...")
df_urban = pd.read_csv(os.path.join(clean_dir, "urbanev_cleaned.csv"))
df_urban['connectionTime'] = pd.to_datetime(df_urban['connectionTime'])

## Feature Engineering for ML
We will create time-based features and lag features.

In [ ]:
# Sort by time
df_urban = df_urban.sort_values(by=['station_id', 'connectionTime'])

df_urban['hour'] = df_urban['connectionTime'].dt.hour
df_urban['dayofweek'] = df_urban['connectionTime'].dt.dayofweek
df_urban['month'] = df_urban['connectionTime'].dt.month
df_urban['is_weekend'] = df_urban['dayofweek'] >= 5

# Target Variables: charger_utilization_rate, kWhDelivered
features = ['hour', 'dayofweek', 'month', 'is_weekend', 'station_id']
targets = ['charger_utilization_rate', 'kWhDelivered']

# Drop NaNs
model_data = df_urban[features + targets].dropna()

X = model_data[features]
y_util = model_data['charger_utilization_rate']
y_load = model_data['kWhDelivered']

# Train-test split (chronological would be better for TS, but standard split for baseline)
X_train, X_test, y_util_train, y_util_test = train_test_split(X, y_util, test_size=0.2, random_state=42)
_, _, y_load_train, y_load_test = train_test_split(X, y_load, test_size=0.2, random_state=42)

## Train XGBoost Models

In [ ]:
print("Training XGBoost for Utilization Rate...")
xgb_util = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
xgb_util.fit(X_train, y_util_train)

print("Training XGBoost for Charging Load (kWh)...")
xgb_load = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
xgb_load.fit(X_train, y_load_train)

## Evaluation Metrics

In [ ]:
def evaluate_model(model, X_test, y_test, name):
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    print(f"--- {name} ---")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE:  {mae:.4f}")
    print(f"R2:   {r2:.4f}")
    return preds, rmse, mae, r2

preds_util, rmse_util, mae_util, r2_util = evaluate_model(xgb_util, X_test, y_util_test, "Utilization Rate")
preds_load, rmse_load, mae_load, r2_load = evaluate_model(xgb_load, X_test, y_load_test, "Charging Load")

# Save outputs
output_metrics = pd.DataFrame({
    'Metric': ['RMSE', 'MAE', 'R2'],
    'Utilization_Rate': [rmse_util, mae_util, r2_util],
    'Charging_Load': [rmse_load, mae_load, r2_load]
})
output_metrics.to_csv(os.path.join(out_dir, "demand_prediction_metrics.csv"), index=False)
print("Metrics saved to demand_prediction_metrics.csv")

# Save Models
joblib.dump(xgb_util, os.path.join(model_dir, "xgb_util.pkl"))
joblib.dump(xgb_load, os.path.join(model_dir, "xgb_load.pkl"))

print("Demand Prediction Agent complete!")